In [ ]:
from __future__ import annotations
import logging
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from tqdm import tqdm

from IPython.display import display

# ===================== 日志配置 =====================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
print('读取分折好的数据')
display(pd.read_csv('/kaggle/input/0-create-fold/folds_by_Species.csv').head())
# ===================== 配置类 =====================
class CFG:
    # 路径相关
    base_path = Path("/kaggle/input/csiro-biomass")
    train_csv = "/kaggle/input/0-create-fold/folds_by_random.csv"
    train_image_dir = base_path / "train"
    output_dir = Path("./models_eva02_single")

    # 模型与图像设置
    model_name = "eva02_base_patch14_448.mim_in22k_ft_in1k"  # timm 模型名
    img_size = 448
    dropout = 0.1
    num_targets = 5  # 直接预测 5 个 target

    # 训练超参数
    batch_size = 8
    num_workers = 2
    num_epochs = 20
    lr = 1e-4
    weight_decay = 1e-5

    # 随机种子
    seed = 42
    # 设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # target 列名（宽表中的顺序）
    target_cols = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g",]
    loss_weights = [0.1, 0.1, 0.1, 0.2, 0.5]  # 对应 target_cols 的顺序
    wandb_project = 'CSIRO_mac14'
CFG.MODEL_ = CFG.model_name.split('.')[0]    
# 创建输出目录
CFG.output_dir.mkdir(parents=True, exist_ok=True)

# 设置随机种子
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

In [ ]:
# ===================== 数据增强接口 =====================
def get_train_transforms(img_size):
    """训练阶段的数据增强（可自行修改）"""
    return A.Compose(
        [
            # 随机水平翻转
            A.HorizontalFlip(p=0.5),
            # 随机垂直翻转
            A.VerticalFlip(p=0.2),
            # 调整到统一尺寸
            A.Resize(img_size, img_size),
            # 标准化
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ]
    )


def get_valid_transforms(img_size):
    """验证阶段的数据预处理（不做随机增强）"""
    return A.Compose(
        [
            A.Resize(img_size, img_size),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ]
    )

In [ ]:
import torch
import torch.nn as nn

class WeightedMSELoss(nn.Module):
    """对多目标回归按列加权的 MSE loss，适配本题 5 个 target"""
    def __init__(self, weights):
        super().__init__()
        # weights: list 或 1D tensor，长度等于 num_targets
        # 注册为 buffer，这样可以自动放到对应 device 上
        self.register_buffer("weights", torch.tensor(weights).view(1, -1))

    def forward(self, preds, targets):
        """
        preds, targets: [batch_size, num_targets]，这里是在 log 空间
        """
        # 计算逐元素平方误差: [B, T]
        diff2 = (preds - targets) ** 2

        # 按列加权: 每一列乘对应 weight
        # weights shape 是 [1, T]，可以广播到 [B, T]
        weighted_diff2 = diff2 * self.weights

        # 这里的平均方式有两种常见写法：
        # 1）直接对所有元素求 mean（等价于 sum / (B * T)）
        loss = weighted_diff2.mean()

        # 如果你想让总权重归一化（严格按“总权重除以 sum(weights)”来算），可以改成：
        # loss = weighted_diff2.sum() / (preds.size(0) * self.weights.sum())

        return loss

class TrainBiomassDataset(Dataset):
    """
    训练数据集：
    - 输入：单流图像
    - 输出：log1p 变换后的 5 个目标值
    """
    def __init__(self, df, image_dir, target_cols, transform):
        """
        df: 宽表，每行一个 image_path + 多个 target 列
        image_dir: 图像所在文件夹路径
        target_cols: 目标列名列表
        transform: Albumentations 数据增强
        """
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.image_paths = self.df["image_path"].values
        self.target_cols = target_cols
        self.transform = transform

        # 从 df 中提取原始 target 数值，类型为 float32
        targets = self.df[self.target_cols].values.astype(np.float32)

        # 对 target 做 log1p 变换（log(1+y)），用于稳定训练
        self.targets_log = np.log1p(targets)

    def __len__(self):
        """返回数据集大小"""
        return len(self.df)

    def __getitem__(self, idx):
        """返回单个样本：图像张量 + log1p 后的目标张量"""

        # 取图像路径
        img_path = self.image_paths[idx]

        # 只使用文件名拼接到 train 目录（保持与你的推理一致）
        full_path = self.image_dir / Path(img_path).name

        # 使用 cv2 读取图像（默认 BGR）
        image = cv2.imread(str(full_path))

        # 如果读取失败，输出警告并使用全 0 图像代替
        if image is None:
            logger.warning(f"Failed to load image: {full_path}, using zeros.")
            image = np.zeros((1000, 2000, 3), dtype=np.uint8)

        # BGR → RGB（符合深度学习模型读取格式）
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 应用 Albumentations 数据增强
        augmented = self.transform(image=image)
        image_tensor = augmented["image"]  # tensor，shape: (3, H, W)

        # 取 log1p 后的目标值
        target_log = self.targets_log[idx]
        target_tensor = torch.from_numpy(target_log)

        return image_tensor, target_tensor
class SimpleBiomassModel(nn.Module):
    """
    单流图像回归模型：
    - 使用 timm backbone 提取图像特征
    - 使用全连接回归头输出 5 个标签的 log 空间预测
    """
    def __init__(self, model_name, num_targets, dropout=0.3):
        super().__init__()

        # 创建 timm backbone，不包含分类头
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,  # 使用预训练权重有助于收敛
            num_classes=0,    # 不要分类头（使用 features 输出）
            global_pool="avg",  # 使用平均池化得到全局向量
        )

        # backbone 输出特征维度
        n_features = self.backbone.num_features

        # 回归头（两层 MLP）
        self.head = nn.Sequential(
            nn.Linear(n_features, n_features // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(n_features // 2, num_targets),
        )

    def forward(self, x):
        """
        前向传播：
        x: 输入图像张量 [B, C, H, W]
        返回：log 空间的预测值 [B, num_targets]
        """
        # 特征提取
        feat = self.backbone(x)

        # 回归输出（log 空间）
        out = self.head(feat)

        return out


In [ ]:
# ===================== 单个 epoch 训练函数（支持 EMA） =====================
def train_one_epoch(model, loader, optimizer, device, ema=None):
    """
    单个 epoch 的训练循环，返回平均 loss（log 空间，加权 MSE）

    参数:
    - model: 当前训练的模型
    - loader: 训练集 DataLoader
    - optimizer: 优化器
    - device: 运行设备（CPU/GPU）
    - ema: 可选的 ModelEMA 实例；如果传入，则在每个 step 后更新 EMA 模型参数
    """
    model.train()
    running_loss = 0.0

    # 在 log 空间使用加权 MSE，对 5 个 target 使用不同权重
    loss_weights = [0.1, 0.1, 0.1, 0.2, 0.5]  # 对应 target_cols 的顺序
    criterion = WeightedMSELoss(loss_weights).to(device)

    for images, targets_log in tqdm(loader, desc="Train", leave=False):
        # images: [B, C, H, W]
        # targets_log: [B, 5]，已经在外面做过 log1p 之类的变换
        images = images.to(device)
        targets_log = targets_log.to(device)

        optimizer.zero_grad()
        preds_log = model(images)          # [B, 5]

        # 计算加权 MSE loss（log 空间）
        loss = criterion(preds_log, targets_log)

        loss.backward()
        optimizer.step()

        # 如果传入了 EMA 对象，则在每个 step 后更新 EMA 模型参数
        if ema is not None:
            ema.update(model)

        # 累加 batch loss（注意这里用 batch_size 加权平均）
        running_loss += loss.item() * images.size(0)

    # 返回整个 epoch 的平均 loss（对所有样本做平均）
    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


# ===================== 验证函数 =====================
def validate(model, loader, device):
    """
    验证循环，返回：
    - avg_loss: 原始空间的加权 MSELoss（自定义监控用）
    - weighted_r2: 原始空间的加权 R²（严格按官方公式）
    """
    model.eval()

    # 验证时的 target 权重（与比赛一致）
    weights = np.array([0.1, 0.1, 0.1, 0.2, 0.5])  # 顺序必须与 target_cols 对齐

    running_loss = 0.0
    all_true = []
    all_pred = []

    with torch.no_grad():
        for images, targets_log in tqdm(loader, desc="Valid", leave=False):
            images = images.to(device)
            targets_log = targets_log.to(device)

            # 模型输出仍然在 log 空间
            preds_log = model(images)

            # 反 log 得到原始空间：y = exp(log_y) - 1
            preds = torch.expm1(preds_log).cpu().numpy()
            trues = torch.expm1(targets_log).cpu().numpy()

            all_pred.append(preds)
            all_true.append(trues)

            # ---------- 原空间加权 MSE（监控用）-----------
            diff2 = (preds - trues) ** 2        # [B, 5]
            weighted_diff2 = diff2 * weights    # 广播到 [B, 5]
            batch_loss = weighted_diff2.mean()  # 对所有元素平均
            running_loss += batch_loss * preds.shape[0]

    # 拼接所有样本
    y_true = np.concatenate(all_true, axis=0)   # [N, 5]
    y_pred = np.concatenate(all_pred, axis=0)   # [N, 5]

    # 对所有样本做平均损失
    avg_loss = running_loss / len(y_true)

    # ---------- 按官方公式计算加权 R² -----------
    weighted_r2 = weighted_r2_score(y_true, y_pred, weights)

    return avg_loss, weighted_r2


# ===================== 加权 R² 计算函数（严格按官方定义） =====================
def weighted_r2_score(y_true, y_pred, weights_per_target):
    """
    y_true, y_pred: [N, T] 的 numpy 数组，T=5
    weights_per_target: 长度为 T 的 1D 数组，例如 [0.1, 0.1, 0.1, 0.2, 0.5]
    按照官方公式计算全局加权 R²。

    数学形式：
        R_w^2 = 1 - sum_j w_j (y_j - ŷ_j)^2 / sum_j w_j (y_j - ȳ_w)^2
        其中 ȳ_w = sum_j w_j y_j / sum_j w_j
    """
    # 转成 numpy 数组，确保是 float
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w_t = np.asarray(weights_per_target, dtype=np.float64)   # [T]

    # 展开成一维：所有 (sample, target) 拼到一起
    N, T = y_true.shape
    y_true_flat = y_true.reshape(-1)   # [N*T]
    y_pred_flat = y_pred.reshape(-1)   # [N*T]

    # 为每个元素生成对应的权重 w_j
    # 把 [T] 的权重按样本数复制 N 次，再展平 -> [N*T]
    w_flat = np.broadcast_to(w_t, (N, T)).reshape(-1)

    # 全局加权均值 ȳ_w
    y_bar_w = np.sum(w_flat * y_true_flat) / (np.sum(w_flat) + 1e-8)

    # 加权残差平方和 SS_res
    ss_res = np.sum(w_flat * (y_true_flat - y_pred_flat) ** 2)

    # 加权总平方和 SS_tot
    ss_tot = np.sum(w_flat * (y_true_flat - y_bar_w) ** 2)

    # 加权 R²
    r2_w = 1.0 - ss_res / (ss_tot + 1e-8)
    return float(r2_w)


# ===================== 手写版本 EMA（适配任意模型） =====================
from copy import deepcopy

class ModelEMA:
    """
    简单稳定的 EMA（指数滑动平均），适配任意 PyTorch 模型

    用法示例：
        ema = ModelEMA(model, decay=0.999)
        ...
        # 每个 step 后
        ema.update(model)
        ...
        # 验证时使用 ema.ema_model
        valid_loss, valid_r2 = validate(ema.ema_model, valid_loader, device)
    """
    def __init__(self, model, decay=0.999):
        # 深拷贝一份模型（EMA 模型不会更新梯度）
        self.ema_model = deepcopy(model)
        self.ema_model.eval()

        # EMA 衰减因子（越接近 1，更新越平滑）
        self.decay = decay

        # EMA 参数不参与梯度计算
        for p in self.ema_model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        """EMA 参数更新：ema = decay * ema + (1 - decay) * model"""
        ema_state = self.ema_model.state_dict()
        model_state = model.state_dict()

        for key in ema_state.keys():
            # ema_param = decay * ema_param + (1-decay) * new_param
            ema_state[key].mul_(self.decay).add_(model_state[key], alpha=1 - self.decay)


In [ ]:
import wandb
import gc
from kaggle_secrets import UserSecretsClient

# ================================
# 5-fold 训练 + wandb + EarlyStopping（简单版）
# ================================
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wandb")
wandb.login(key=secret_value_0)

for csv_path in [
    '/kaggle/input/0-create-fold/folds_by_random.csv',
    '/kaggle/input/0-create-fold/folds_by_Pre_GSHH_NDVI.csv',
    '/kaggle/input/0-create-fold/folds_by_Sampling_Date.csv',
    '/kaggle/input/0-create-fold/folds_by_Species.csv',
]:
    CFG.train_csv = csv_path
    Group_mode = csv_path.split('/')[-1].split('.')[0].split('_')[-1]

    train_wide = pd.read_csv(CFG.train_csv)
    fold_scores = []

    for fold_idx in range(5):

        # ============================
        # wandb 初始化
        # ============================
        run = wandb.init(
            project=CFG.wandb_project,
            name=f"{Group_mode}_{CFG.MODEL_}_fold{fold_idx}_size{CFG.img_size}_bs{CFG.batch_size}_lr{CFG.lr}",
            config={
                "Group_mode": Group_mode,
                "model_name": CFG.model_name,
                "fold_idx": fold_idx,
                "image_size": int(CFG.img_size),
                "batch_size": int(CFG.batch_size),
                "lr": float(CFG.lr),
                "num_epochs": int(CFG.num_epochs),
            },
            reinit=True,
        )

        # ============================
        # 划分训练 / 验证
        # ============================
        df_train = train_wide[train_wide["fold"] != fold_idx].reset_index(drop=True)
        df_valid = train_wide[train_wide["fold"] == fold_idx].reset_index(drop=True)

        train_transform = get_train_transforms(CFG.img_size)
        valid_transform = get_valid_transforms(CFG.img_size)

        train_dataset = TrainBiomassDataset(
            df=df_train,
            image_dir=CFG.train_image_dir,
            target_cols=CFG.target_cols,
            transform=train_transform,
        )
        valid_dataset = TrainBiomassDataset(
            df=df_valid,
            image_dir=CFG.train_image_dir,
            target_cols=CFG.target_cols,
            transform=valid_transform,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=CFG.batch_size,
            shuffle=True,
            num_workers=CFG.num_workers,
            pin_memory=True,
        )
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=CFG.batch_size,
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=True,
        )

        # ============================
        # 模型与优化器
        # ============================
        model = SimpleBiomassModel(
            model_name=CFG.model_name,
            num_targets=len(CFG.target_cols),
            dropout=CFG.dropout,
        ).to(CFG.device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CFG.lr,
            weight_decay=CFG.weight_decay,
        )

        # ============================
        # Early Stopping
        # ============================
        patience = 5
        es_counter = 0
        best_r2 = -1e9
        best_epoch = 0
        best_model_path = CFG.output_dir / f"{Group_mode}_best_model_fold{fold_idx}.pth"

        # ============================
        # 训练循环（朴素版）
        # ============================
        for epoch in range(1, CFG.num_epochs + 1):

            logger.info(f"[{Group_mode}] Fold {fold_idx} Epoch {epoch}/{CFG.num_epochs}")

            # 训练
            train_loss = train_one_epoch(model, train_loader, optimizer, CFG.device)

            # 验证
            valid_loss, valid_r2 = validate(model, valid_loader, CFG.device)

            print(
                f"[{Group_mode}] Fold {fold_idx} | Epoch {epoch} | "
                f"Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f} | "
                f"Valid R2: {valid_r2:.4f}"
            )

            # wandb 记录
            wandb.log(
                {
                    "epoch": epoch,
                    "fold": fold_idx,
                    "Group_mode": Group_mode,
                    "train_loss": train_loss,
                    "valid_loss": valid_loss,
                    "valid_r2": valid_r2,
                }
            )

            # ============================
            # EarlyStopping逻辑
            # ============================
            if valid_r2 > best_r2:
                best_r2 = valid_r2
                best_epoch = epoch
                es_counter = 0
                torch.save(model.state_dict(), best_model_path)
                wandb.run.summary["best_valid_r2"] = best_r2
                wandb.run.summary["best_epoch"] = best_epoch
                print(f"  ✓ New best model saved: {best_model_path}")
            else:
                es_counter += 1
                print(f"  ✗ No improvement ({es_counter}/{patience})")

            if es_counter >= patience:
                print(f"  🔥 Early stopping triggered at epoch {epoch}")
                break

        print(f"[{Group_mode}] Fold {fold_idx} Best R2: {best_r2:.4f} at epoch {best_epoch}")
        fold_scores.append(best_r2)

        wandb.finish()

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ============================
    # 汇总
    # ============================
    print(f"\n========== 5-FOLD RESULTS ({Group_mode}) ==========")
    for i, score in enumerate(fold_scores):
        print(f"Fold {i} R2: {score:.4f}")

    print(f"\n⭐ Average R2 ({Group_mode}): {np.mean(fold_scores):.4f}\n")
